# DurianAI — TF.js Export Only

This notebook **skips training** and directly loads the pre-trained vision model to export TF.js format.

**Estimated time: 2 minutes** (vs 30 minutes for full training)

In [ ]:
#@title 1. Install dependencies { display-mode: "form" }
!pip install -q tensorflow tensorflowjs numpy pillow scikit-learn

import tensorflow as tf
from tensorflow.keras import layers, applications
import numpy as np
import os
print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
#@title 2. Build and train the vision model (same architecture as DurianAI) { display-mode: "form" }
IMG_SIZE = 224

# Download data from original source (same as full notebook)
ROBOFLOW_API_KEY = 'f5G80KlyZfhoBRL8iynH' #@param {type:"string"}

DATA_DIR = '/content/durian_data'
os.makedirs(DATA_DIR, exist_ok=True)

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace('durian-cnn').project('durian-ripeness-detection-xtned')
    dataset = project.versions()[0].download('folder', location=DATA_DIR)
    
    # Also download mutruity dataset
    try:
        project2 = rf.workspace('wjy-tis6h').project('durian_mutruity')
        dataset2 = project2.versions()[0].download('multiclass', location='/content/durian_mutruity')
    except Exception as e:
        print(f'mutruity optional: {e}')
    print('Data downloaded!')

# Load images (adapted from full notebook)
from PIL import Image
from collections import Counter

UNIFIED_LABELS = ['unripe', 'ripe', 'overripe']

def load_image_folder(base_dir, split='train', label_mapping=None):
    X, y = [], []
    split_dir = os.path.join(base_dir, split)
    if not os.path.isdir(split_dir):
        split_dir = os.path.join(base_dir, 'valid' if split == 'val' else split)
    if not os.path.isdir(split_dir):
        return np.array(X), np.array(y)
    for cls_dir in os.listdir(split_dir):
        cls_path = os.path.join(split_dir, cls_dir)
        if not os.path.isdir(cls_path):
            continue
        mapped_label = cls_dir.lower()
        if label_mapping and cls_dir in label_mapping:
            mapped_label = label_mapping[cls_dir]
        if mapped_label not in UNIFIED_LABELS:
            continue
        for img_file in os.listdir(cls_path):
            if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            try:
                img = Image.open(os.path.join(cls_path, img_file)).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
                X.append(np.array(img, dtype=np.float32) / 255.0)
                y.append(mapped_label)
            except:
                pass
    return np.array(X), np.array(y)

X_train, y_train = load_image_folder(DATA_DIR, 'train', {'Ripe': 'ripe', 'Unripe': 'unripe', 'Defect': 'defect'})
X_val, y_val = load_image_folder(DATA_DIR, 'valid', {'Ripe': 'ripe', 'Unripe': 'unripe', 'Defect': 'defect'})

# Also try mutruity
if os.path.isdir('/content/durian_mutruity'):
    Xm, ym = load_image_folder('/content/durian_mutruity', 'train', {'immature': 'unripe', 'mature': 'ripe'})
    if len(Xm) > 0:
        X_train = np.concatenate([X_train, Xm]) if len(X_train) > 0 else Xm
        y_train = np.concatenate([y_train, ym]) if len(y_train) > 0 else ym

print(f'Train: {len(X_train)} images')
print(f'Distribution: {dict(Counter(y_train))}')

label_map = {name: i for i, name in enumerate(UNIFIED_LABELS[:2])}  # ripe/unripe for current model
y_train_enc = np.array([label_map[l] for l in y_train if l in label_map])
X_train = np.array([X_train[i] for i, l in enumerate(y_train) if l in label_map])
print(f'Final training: {X_train.shape}')

In [ ]:
#@title 3. Build MobileNetV2 and train { display-mode: "form" }
BATCH_SIZE = 32

base_model = applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(2, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.fit(
    X_train, y_train_enc,
    validation_data=(X_val, np.array([label_map[l] for l in y_val if l in label_map])),
    epochs=30, batch_size=BATCH_SIZE,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1,
)
print('Training complete!')

In [ ]:
#@title 4. Export TF.js for browser deployment { display-mode: "form" }
import tensorflowjs as tfjs

tfjs_dir = '/content/durian_vision_tfjs'
os.makedirs(tfjs_dir, exist_ok=True)

print('Converting to TF.js format...')
tfjs.converters.save_keras_model(model, tfjs_dir)

# List output files
for f in os.listdir(tfjs_dir):
    size = os.path.getsize(os.path.join(tfjs_dir, f))
    print(f'  {f}: {size/1024:.1f} KB')

# Add labels
with open(os.path.join(tfjs_dir, 'labels.txt'), 'w') as f:
    f.write('ripe\nunripe\n')

print(f'\nTF.js model ready at: {tfjs_dir}')
print('Download these files and place them in frontend/public/models/vision/')

In [ ]:
#@title 5. Download All Results { display-mode: "form" }
from google.colab import files
import os

for f in os.listdir(tfjs_dir):
    filepath = os.path.join(tfjs_dir, f)
    if os.path.getsize(filepath) < 100 * 1024 * 1024:
        print(f'Downloading: {f}')
        files.download(filepath)

print('\n=== Deployment Instructions ===')
print('1. Copy all downloaded files to: frontend/public/models/vision/')
print('2. cd durian-app && git add frontend/public/models/vision/')
print('3. git commit -m "deploy: vision TF.js model" && git push')
print('4. Render auto-deploys → App vision switches to AI mode!')